# Set up

For this challenge you will have access to the Swiss AI Research Platform and models served from there. Models available can be found [here](https://serving.swissai.svc.cscs.ch/)

For a rundown on models you have available to you:

`swiss-ai/Apertus-8B-Instruct-2509` -- Small version of Apertus, the open model developed by the EPFL and ETH Zurich

`swiss-ai/Apertus-70B-Instruct-2509` -- Large version of Apertus.

`zai-org/GLM-4.7-Flash` -- Small Mixture of Experts model (fast, but somewhat less performant)

`moonshotai/Kimi-K2.5-SDSC` -- Most performant model you have available for this hackathon. Vision capable (you may attach images with your prompts.)

In [1]:
import os
import base64
import requests
import cairosvg
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from a .env file if present

True

In [2]:
# Initial client setup
client_cscs = OpenAI(
    base_url="https://api.swissai.svc.cscs.ch/v1",  # This is the endpoint you will be using to access the API
    api_key=os.getenv("CSCS_API_KEY")               # You will have to set this in your environment, or provie a .env file for load_dotenv to work
)

# Execution

## Text Only

### Create ask Apertus function

In [3]:
# Apertus comes in 8B and 70B variants. Change the string to match your endpoint's model name
def ask_apertus(prompt: str, model: str = "swiss-ai/Apertus-8B-Instruct-2509") -> str:
    try:
        response = client_cscs.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.5
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

### Test Apertus and LLM with prompts

In [4]:
# Test Apertus
prompt_apertus = "Explain why digital sovereignty matters for Switzerland."
print(f"Prompt: {prompt_apertus}\n")

print(ask_apertus(prompt_apertus))

Prompt: Explain why digital sovereignty matters for Switzerland.

Digital sovereignty refers to the ability of a country to control and manage its digital infrastructure and data within its territory. For Switzerland, digital sovereignty is particularly important for several reasons:

1. Data Protection: Switzerland has a strong tradition of protecting personal data and ensuring the privacy of its citizens. Digital sovereignty allows Switzerland to enforce its own data protection laws and regulations, preventing the transfer of sensitive data to countries with less stringent data protection laws.

2. Economic Competitiveness: Switzerland is a hub for international trade and finance. Digital sovereignty ensures that Switzerland can maintain its competitive edge in the global digital economy by controlling its own digital infrastructure and data.

3. National Security: Digital sovereignty is critical for national security. By controlling its digital infrastructure and data, Switzerland c

## With Image

### Create Function

In [ ]:

def encode_image_from_url(url: str) -> str:
    r = requests.get(url)
    r.raise_for_status()
    mime = r.headers.get("content-type", "image/svg+xml").split(";")[0]
    # SVG has a speciic requirement to be rasterized first. You should not have to do this with regular PNG or JPG images.
    image_data = r.content if mime != "image/svg+xml" else cairosvg.svg2png(bytestring=r.content)
    data = base64.b64encode(image_data).decode("utf-8")
    return f"data:{mime};base64,{data}"

def ask_with_image(prompt: str, image_path: str, model: str = "moonshotai/Kimi-K2.5-SDSC") -> str:
    encoded_image = encode_image_from_url(image_path)
    try:
        response = client_cscs.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": encoded_image}},
                ]},
            ],
            temperature=0.5
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

In [6]:
image_url = "https://cdn.prod.website-files.com/63ea083f4e797cf53055586b/69df9c31885816580fe4151c_SDSC_Logo_RGB.svg"
prompt_image = "Describe what is in this image."
print(ask_with_image(prompt_image, image_url))

This image shows the **logo of SDSC** (San Diego Supercomputer Center). 

The logo consists of two main elements:

1. **Icon**: On the left, there's a stylized triangular symbol made of three curved, interlocking ribbon-like segments forming a continuous loop. The segments are colored in different shades: **green** (right side), **medium blue** (left side), and **dark blue/purple** (bottom).

2. **Text**: To the right of the icon, the letters **"SDSC"** appear in bold, dark blue/purple sans-serif typography.

The design has a modern, clean look with a transparent/white background, and the color scheme uses various shades of blue and green, suggesting technology, innovation, and data connectivity.


## Embedding

You may also require an embedding model. One of these is available on the compute platform. Namely: `Snowflake/snowflake-arctic-embed-l-v2.0`

In [7]:
def embed_text(text: str, model: str = "Snowflake/snowflake-arctic-embed-l-v2.0") -> list:
    try:
        response = client_cscs.embeddings.create(
            model=model,
            input=text
        )
        return response.data[0].embedding
    except Exception as e:
        return f"An error occurred: {e}"

In [10]:
sample_text = "Switzerland is a country in Europe known for its mountains and lakes."
embedding = embed_text(sample_text)
print(f"Embedding for sample text: {embedding[:5]}...")
print(f"Embedding length: {len(embedding)}")

Embedding for sample text: [0.04903201013803482, -0.03163592144846916, -0.04308651387691498, -0.07773188501596451, 0.02205706387758255]...
Embedding length: 1024
